# WBSC 검수

`WBSC 크롤링.ipynb`가 만든 **PBP 파일 자체**를 공식 boxScore와 대조해서 diff 파일을 만든다.

- **실제값** (읽기만 함): `inning_summary_global_{YEAR}.xlsx`, `batter_box_global_{YEAR}.xlsx`,
  `pitcher_box_global_{YEAR}.xlsx` — WBSC 공식 boxScore/gameData 그대로.
- **sim값** (여기서 계산): `wbsc/{YYYY}_{MM}_global_{letter}.xlsx`(KBSA 포맷 PBP)의
  한글 `result` 코드 / `outs_when_up` / `away_score`·`home_score` / `order` 에서 재도출.

> ⚠ **예전 버전과 달라진 점**: 이전 검수는 sim값을 WBSC 원본의 구조화 필드(`ab`/`h`/`rbi`/`runs`)에서
> 다시 집계했다. 실제값도 같은 원본이라 사실상 *WBSC를 WBSC와 비교*하는 구조였고, 정작 산출물인
> PBP는 한 줄도 검사하지 않았다. 그래서 아웃 42개가 빠지고 유령 half가 있고 타자가 오배정된
> 경기들에도 전부 "불일치 0행"을 보고했다. 지금은 PBP 파일만 읽어서 검사한다 — 네트워크 요청도 없다.

만드는 파일:
- `inning_summary_diff_global_{YEAR}.xlsx` — 이닝별 아웃/득점
- `batter_diff_global_{YEAR}.xlsx` — 타자 AB/H/RBI (한글 코드에서 재도출)
- `pitcher_diff_global_{YEAR}.xlsx` — 투수 BF
- `review_confidence_global_{YEAR}.xlsx` — 번역 신뢰도 med/low/none 행

**전제**: `WBSC 크롤링.ipynb`를 먼저 돌려서 위 실제값 3개와 PBP 파일들이 있어야 한다.

주자상황·득점자(R)·승계주자 책임투수(9.16⒢)까지 완전히 검증하려면 이 PBP 파일을
"2. 아웃카운트 및 점수 체크" / "3. 주자상황 체크" 노트북에 그대로 넣어 돌린다.

In [ ]:
"""
검수 셋업 — 네트워크 요청 없이 로컬 파일만 읽는다.

읽는 것:
  - PBP(산출물):  wbsc/{YYYY}_{MM}_global_{letter}.xlsx   <- 이게 진짜 검수 대상
  - 실제값(공식): wbsc/inning_summary_global_{YEAR}.xlsx
                  wbsc/batter_box_global_{YEAR}.xlsx
                  wbsc/pitcher_box_global_{YEAR}.xlsx
"""
import re, pathlib
import pandas as pd

OUT = pathlib.Path("wbsc")
YEAR = 2026   # *_global_{YEAR}.xlsx 파일명에 쓸 연도 (크롤링 노트북과 동일해야 함)

# ── KBSA 결과코드 규칙 (3. 주자상황 체크.ipynb 와 동일하게 유지할 것) ──────────
WALK_RESULTS = {'4구', '사구', '고의4구', '고의4구[자동]', '4구.폭투', '타격방해'}
SAC_OUT = {
    '우희플', '좌희플', '중희플',
    '좌파희플', '우파희플', '2파희플', '1파희플', '3파희플', '유파희플',
    '1희번', '3희번', '투희번', '포희번', '2희번', '유희번',
}
# 국가대표(WBSC) 전용: 실책을 저지른 자리까지 남긴 코드 — '투송실'(투수 송구실책),
# '포포실'(포수 포구실책). KBSA 는 자리 없이 '송구실책'/'포구실책' 으로만 적으므로
# 이 코드가 KBSA 자료에 나오는 일은 없다. 부분일치 검사에 '포실' 만 넣으면 KBSA 의
# '포실'(포수 실책 출루)까지 걸리므로, 반드시 아홉 자리를 모두 적은 집합을 쓴다.
_NT_POS = ('투', '포', '1', '2', '3', '유', '좌', '중', '우')
NT_ERRORS = {f'{_p}{_k}' for _p in _NT_POS for _k in ('송실', '포실')}
# 국가대표(WBSC) 전용: 번트 병살에 번트임을 넣은 코드('포번병살'). KBSA 는
# '투땅병살[BUNT]' 처럼 태그로 적어 어휘에는 없다. 병살·타수 판정은 부분일치라
# 집계는 '포땅병살' 과 똑같이 잡힌다.
NT_BUNT_DP = {f'{_p}번병살' for _p in _NT_POS}
NON_PA = {
    *NT_ERRORS,
    '대주자', '대타', '도루', '도루자', '폭투', '보크', '포일',
    '주자아웃', '부정주루', '견제사', '승부주자', '송구실책', '포구실책', '낙구', '실책',
    '주루방해', '타구맞음', '홈공과어필아웃', '런다운', '오버런', '득점', '주자이동',
    '리터치',
}
OUT_EVENTS = {'도루자', '주자아웃', '견제사', '수비방해', '루공과', '타구맞음',
              '홈공과어필아웃', '런다운', '오버런', '리터치', '부정주루'}


def _clean(result):
    """3. 주자상황 체크의 _sim_clean_result 축약판 (WBSC PBP에는 쉼표/대괄호 표기가 없다)."""
    return '' if pd.isna(result) else str(result).split(',')[0].strip()


def is_ab(result) -> bool:
    """타수(AB)로 세는 결과인지. 볼넷/희생타/승부주자는 AB가 아니다."""
    c = _clean(result)
    if not c or c in WALK_RESULTS or c in SAC_OUT or c in NON_PA:
        return False
    if c.endswith('번출') or c.endswith('플출') or c.endswith('희플') or c.endswith('희번'):
        return False
    if c == '승부주자':
        return False
    return True


def is_hit(result) -> bool:
    c = _clean(result)
    if not c or c in OUT_EVENTS or c in NON_PA:
        return False
    return c.endswith('안') or c.endswith('2') or c.endswith('3') or '홈' in c


def load_pbp(year: int) -> pd.DataFrame:
    """크롤링 노트북이 만든 PBP 파일을 전부 읽어 하나로 합친다."""
    files = [f for f in sorted(OUT.glob("*_global_[a-z].xlsx"))
             if re.match(rf'^{year}_\d{{2}}_global_[a-z]\.xlsx$', f.name)]
    if not files:
        return pd.DataFrame()
    frames = []
    for f in files:
        d = pd.read_excel(f)
        d["file_name"] = f.stem
        frames.append(d)
    return pd.concat(frames, ignore_index=True)


def load_actual(year: int):
    def _read(name):
        p = OUT / f"{name}_{year}.xlsx"
        return pd.read_excel(p) if p.exists() else pd.DataFrame()
    return (_read("inning_summary_global"), _read("batter_box_global"), _read("pitcher_box_global"))


pbp = load_pbp(YEAR)
actual_inning, actual_batter, actual_pitcher = load_actual(YEAR)
print(f"PBP: {len(pbp)}행 / {pbp['game_id'].nunique() if len(pbp) else 0}경기")
print(f"실제값: inning {len(actual_inning)}행 / batter {len(actual_batter)}행 / "
      f"pitcher {len(actual_pitcher)}행")
if len(pbp):
    print("PBP 행 종류:", pbp['result'].isin(NON_PA).map({True: '비타석', False: '타석'})
          .value_counts().to_dict())


In [ ]:
"""
검수 함수들.

**중요**: sim 값은 반드시 PBP 파일(산출물)에서 계산한다. WBSC 원본의 구조화 필드
(ab/h/rbi/runs)를 다시 집계하면 "WBSC를 WBSC와 비교"하는 셈이 되어, 정작 산출물인
KBSA 포맷 PBP의 오류(한글 결과코드 오역, 비타석 행 누락, 유령 half 등)를 하나도
못 잡는다 — 실제로 이전 버전이 그런 구조였고, 아웃 42개 누락/유령 half/타자 오배정이
있는 경기들에 대해 전부 "불일치 0행"을 보고했다.

검사 항목
  A. 결과코드 어휘 정합성 : KBSA 실데이터 203개 코드에 없는 코드
  B. 번역 신뢰도          : result_confidence != high
  C. 타자 이름 정합성     : result_en(영문 원문)이 batter 이름으로 시작하는지
  D. 타순 순환            : 팀별 order가 1~9로 순환하는지
  E. 이닝별 아웃/득점     : half별 outs_when_up/점수 합 vs inning_summary
  F. 타자 성적            : 한글 result 코드에서 재도출한 AB/H/RBI vs 공식 boxScore
  G. 투수 상대타자수      : PBP 행에서 센 BF vs 공식 boxScore
  H. 한국 선수 이름       : 한글로 치환되지 않고 로마자로 남은 선수
  I. 타점·득점 원문 대조  : 영문 원문이 밝힌 타점/득점자 수 vs PBP 열

F 의 RBI 만은 성격이 다르다. AB/H 는 한글 코드에서 다시 도출하니 번역기를
검증하지만, RBI 는 WBSC 의 플레이별 rbi 를 더해 WBSC 의 boxScore rbi 와 맞추는
것이라 둘 다 출처가 같다 — 타자 오배정·행 중복은 잡아도 값 자체는 검증하지
못한다. 그 자리를 I 가 메운다.

주자상황·득점자(R)·승계주자 책임투수까지 완전히 검증하려면 이 PBP 파일을
"2. 아웃카운트 및 점수 체크" / "3. 주자상황 체크" 노트북에 그대로 넣어 돌리면 된다
(2025 대회 기준 batter_box AB/H/R/RBI 0불일치 / 이닝 아웃 0불일치까지 확인됨).
"""

KBSA_VOCAB = {
    '1내안', '1땅', '1땅R', '1땅병살', '1땅실', '1번', '1번R', '1번안', '1실', '1야선', '1인플', '1직', '1직병살',
    '1파플', '1플', '1희번', '1희번출', '2내안', '2땅', '2땅R', '2땅병살', '2땅실', '2번안', '2실', '2야선', '2인플',
    '2직', '2파플', '2플', '2플실', '2희플', '3내안', '3땅', '3땅R', '3땅병살', '3땅실', '3번', '3번R', '3번안',
    '3실', '3야선', '3인플', '3직', '3직병살', '3직실', '3파플', '3플', '3희번', '3희번출', '4구', '견제사', '고의4구',
    '낫아웃+', '낫아웃+포일', '낫아웃+폭투', '낫아웃-', '대수비', '대주자', '대타', '도루', '도루자', '득점', '루공과', '보크',
    '사구', '삼진', '송구실책', '수비방해', '승부주자', '실책', '쓰리번트', '우땅', '우땅R', '우선2', '우선3', '우선안', '우실',
    '우안', '우안2', '우안3', '우월2', '우월3', '우월G홈', '우월안', '우월홈', '우전2', '우전3', '우전안', '우중2', '우중3',
    '우중G홈', '우중안', '우중홈', '우직', '우파플', '우파희플', '우플', '우플실', '우희플', '유내안', '유땅', '유땅R', '유땅병살',
    '유땅실', '유실', '유야선', '유인플', '유직', '유파플', '유플', '유희번', '유희플', '인좌선2', '인좌월2', '인중월2', '좌G홈',
    '좌땅', '좌땅R', '좌선2', '좌선3', '좌선안', '좌실', '좌안', '좌안2', '좌안3', '좌월2', '좌월3', '좌월G홈', '좌월안',
    '좌월홈', '좌전2', '좌전3', '좌전안', '좌중2', '좌중3', '좌중G홈', '좌중안', '좌중홈', '좌직', '좌파플', '좌파희플', '좌플',
    '좌플실', '좌희플', '좌희플출', '주루방해', '주자아웃', '중땅', '중땅R', '중실', '중안', '중안2', '중안3', '중월2', '중월3',
    '중월G홈', '중월안', '중월홈', '중전2', '중전3', '중전안', '중직', '중플', '중플실', '중희플', '중희플출', '타격방해',
    '타구맞음', '타자타구', '투내안2', '투땅', '투땅R', '투땅병살', '투번', '투번R', '투번안', '투실', '투안', '투앞R', '투야선',
    '투인플', '투직', '투파플', '투플', '투희번', '투희번출', '포구실책', '포땅', '포땅R', '포땅병살', '포번', '포번R', '포번안',
    '포실', '포안', '포야선', '포인플', '포일', '포파플', '포플', '포희번', '포희번출', '폭투',
}
# 어휘에는 없지만 구조상 정상인 코드 (해당 방향/포지션 조합이 KBSA 데이터에 우연히
# 없었을 뿐). 새로 나오면 실물 확인 후 여기에 추가하거나 KBSA_VOCAB을 갱신한다.
VOCAB_ALLOWLIST = {'투내안', '중야선', '우야선'} | NT_ERRORS | NT_BUNT_DP


def check_vocab(pbp):
    """A. KBSA 실데이터 어휘에 없는 결과코드."""
    bad = pbp[~pbp['result'].isin(KBSA_VOCAB | VOCAB_ALLOWLIST)]
    return bad[['file_name', 'game_id', 'team', 'inning', 'topbot', 'order', 'batter',
                'result', 'result_en', 'result_confidence']]


def check_confidence(pbp):
    """B. 번역 신뢰도가 high가 아닌 행."""
    bad = pbp[pbp['result_confidence'] != 'high']
    return bad[['file_name', 'game_id', 'team', 'inning', 'topbot', 'result',
                'result_en', 'result_confidence']]


def check_batter_name(pbp):
    """C. 영문 원문(result_en)이 batter 이름으로 시작하는지.

    WBSC의 batterid는 같은 타순 교체 직후 한 타석에서 이전 선수 값으로 남는 버그가
    있어서(2025 대회 2건), 이름 대조는 그걸 잡아내는 가장 값싼 검사다.
    한글로 치환된 한국 선수는 대조가 불가능하므로 제외한다."""
    # 승부주자/대주자 행은 result_en이 배치·교체 안내문이라 이름 대조 대상이 아니다
    d = pbp[pbp['batter'].notna() & pbp['plate_seq'].notna()
            & ~pbp['result'].isin(NON_PA)].copy()
    d = d[d['batter'].astype(str).str.isascii()]
    ok = pd.Series([str(en).strip().startswith(str(bt))
                    for en, bt in zip(d['result_en'], d['batter'])], index=d.index)
    bad = d[~ok]
    return bad[['file_name', 'game_id', 'team', 'inning', 'topbot', 'order', 'batter', 'result_en']]


def check_order_rotation(pbp):
    """D. 팀별 타순이 1~9로 순환하는지 (타석 행만)."""
    # 승부주자는 타순 순환을 따르지 않으므로(직전 타자들이 루상에 배치된다) 제외
    rows = []
    d = pbp[pbp['plate_seq'].notna() & ~pbp['result'].isin(NON_PA)]
    for (gid, team), grp in d.groupby(['game_id', 'team']):
        seq = grp.sort_values('plate_seq')
        prev = None
        for _, r in seq.iterrows():
            cur = int(r['order'])
            if prev is not None and cur != (prev % 9) + 1:
                rows.append({'game_id': gid, 'team': team, 'inning': r['inning'],
                             'topbot': r['topbot'], 'prev_order': prev, 'order': cur,
                             'batter': r['batter']})
            prev = cur
    return pd.DataFrame(rows)


def check_inning(pbp, actual_inning):
    """E. half별 아웃/득점 합 vs inning_summary(공식)."""
    d = pbp.copy()
    d['_out'] = pd.to_numeric(d['outs_when_up'], errors='coerce').fillna(0)
    d['_run'] = (pd.to_numeric(d['away_score'], errors='coerce').fillna(0)
                 + pd.to_numeric(d['home_score'], errors='coerce').fillna(0))
    g = (d.groupby(['game_id', 'team', 'topbot', 'inning'])[['_out', '_run']]
         .sum().reset_index()
         .rename(columns={'_out': 'pbp_out', '_run': 'pbp_score'}))
    m = actual_inning.merge(g, on=['game_id', 'team', 'topbot', 'inning'], how='outer',
                            indicator=True).fillna(0)
    m[['pbp_out', 'pbp_score']] = m[['pbp_out', 'pbp_score']].astype(int)
    m['out_diff'] = m['pbp_out'] - m['real_out']
    m['score_diff'] = m['pbp_score'] - m['score']
    cols = ['game_id', 'team', 'topbot', 'inning',
            'score', 'pbp_score', 'score_diff', 'real_out', 'pbp_out', 'out_diff', '_merge']
    return m[cols]


def check_batter_box(pbp, actual_batter):
    """F. 한글 result 코드에서 재도출한 AB/H/RBI vs 공식 boxScore.

    이게 번역기(classify_result)를 실제로 검증하는 유일한 검사다 — WBSC 원본의
    ab/h/rbi 필드를 다시 더하면 번역이 아무리 틀려도 항상 일치해버린다."""
    d = pbp[pbp['plate_seq'].notna() & pbp['batter'].notna()].copy()
    d = d[~d['result'].isin(NON_PA)]
    d['_ab'] = d['result'].apply(is_ab)
    d['_h'] = d['result'].apply(is_hit)
    d['_rbi'] = pd.to_numeric(d['RBI'], errors='coerce').fillna(0)
    sim = (d.groupby(['game_id', 'team', 'batter'])
           .agg(sim_PA=('result', 'size'), sim_AB=('_ab', 'sum'),
                sim_H=('_h', 'sum'), sim_RBI=('_rbi', 'sum')).reset_index())
    m = actual_batter.merge(sim, on=['game_id', 'team', 'batter'], how='left')
    for c in ('sim_PA', 'sim_AB', 'sim_H', 'sim_RBI'):
        m[c] = m[c].fillna(0).astype(int)
    m['AB_diff'] = m['sim_AB'] - m['AB']
    m['H_diff'] = m['sim_H'] - m['H']
    m['RBI_diff'] = m['sim_RBI'] - m['RBI']
    cols = ['game_id', 'team', 'uniform', 'batter', 'PA', 'sim_PA',
            'AB', 'sim_AB', 'AB_diff', 'H', 'sim_H', 'H_diff', 'RBI', 'sim_RBI', 'RBI_diff']
    return m[cols]


def check_pitcher_box(pbp, actual_pitcher):
    """G. PBP에서 센 상대타자수(BF) vs 공식 boxScore.

    승부치기 주자(승부주자) 행은 제외한다 — KBSA 관례는 그 주자를 타석으로 세지만
    (기록지에 타자 이름/타석 순번이 붙는다) WBSC의 pitch_bf는 세지 않아서, 넣으면
    승부치기 half 1개당 BF가 2씩 부풀어 실제 오류가 아닌 불일치가 생긴다.

    실점(R)/자책(ER)의 승계주자 책임 배분(9.16⒢)은 여기서 검증하지 않는다 —
    "3. 주자상황 체크"의 compare_pitcher_box(부채 큐 모델)를 써야 한다."""
    d = pbp[pbp['plate_seq'].notna() & (pbp['result'] != '승부주자')]
    # PBP의 def_team(수비팀) = 투수 소속팀. 공식 boxScore는 team이 투수 소속팀이고
    # def_team이 상대팀이라 뜻이 반대라, 이름 충돌을 피하려고 미리 떼어낸다.
    sim = d.groupby(['game_id', 'def_team', 'pitcher']).size().reset_index(name='sim_BF')
    a = actual_pitcher.drop(columns=['def_team'], errors='ignore').rename(
        columns={'team': '_pteam'}).copy()
    m = a.merge(sim, left_on=['game_id', '_pteam', 'pitcher'],
                right_on=['game_id', 'def_team', 'pitcher'], how='outer', indicator=True)
    m['team'] = m['_pteam'].fillna(m['def_team'])
    m['sim_BF'] = m['sim_BF'].fillna(0).astype(int)
    m['BF'] = pd.to_numeric(m['BF'], errors='coerce').fillna(0).astype(int)
    m['BF_diff'] = m['sim_BF'] - m['BF']
    return m[['game_id', 'team', 'uniform', 'pitcher', 'BF', 'sim_BF', 'BF_diff', 'R', 'ER', '_merge']]


def save_diff_xlsx(df, path, diff_cols):
    """KBSA save_box_diff_xlsx()와 동일한 스타일: diff==0 초록 / !=0 빨강."""
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

    green_f = PatternFill('solid', start_color='C6EFCE', end_color='C6EFCE')
    red_f = PatternFill('solid', start_color='FFD7D7', end_color='FFD7D7')
    thin = Side(style='thin', color='BFBFBF')
    bdr = Border(left=thin, right=thin, top=thin, bottom=thin)
    hf = Font(name='Arial', bold=True, color='FFFFFF', size=10)
    df_f = Font(name='Arial', size=10)
    ctr = Alignment(horizontal='center', vertical='center')
    hfill = PatternFill('solid', start_color='4472C4', end_color='4472C4')

    wb = Workbook(); ws = wb.active
    headers = [c for c in df.columns if c != '_merge']
    for c, h in enumerate(headers, 1):
        cell = ws.cell(1, c, h)
        cell.font = hf; cell.fill = hfill; cell.alignment = ctr; cell.border = bdr
    for ri, row in df.reset_index(drop=True).iterrows():
        er = ri + 2
        for c, h in enumerate(headers, 1):
            val = row[h]
            cell = ws.cell(er, c, val if not pd.isna(val) else '')
            cell.font = df_f; cell.alignment = ctr; cell.border = bdr
        for dc in diff_cols:
            if dc in headers:
                ci = headers.index(dc) + 1
                ws.cell(er, ci).fill = green_f if row[dc] == 0 else red_f
    for ci in range(1, len(headers) + 1):
        ws.column_dimensions[ws.cell(1, ci).column_letter].width = 14
    ws.freeze_panes = 'A2'
    wb.save(path)


print("검수 함수 정의 완료")


def check_korea_names(pbp):
    """H. 한국 선수인데 한글로 치환되지 않고 로마자로 남은 이름.

    KOREA_NAME_MAP 에 없는 선수는 조용히 로마자로 흘러가 batter_box 와 웹까지
    그대로 간다 — 2025 때는 이걸 손으로 챙겼다. 대표팀 명단이 바뀌거나 WBSC 의
    표기가 매핑 키와 다르면(띄어쓰기·철자) 여기서 잡힌다."""
    d = pbp[pbp['team'].astype(str).str.contains('Korea', case=False, na=False)
            & pbp['batter'].notna()]
    bad = d[d['batter'].astype(str).str.isascii()]
    if bad.empty:
        return pd.DataFrame(columns=['game_id', 'team', 'batter', '행수'])
    return (bad.groupby(['game_id', 'team', 'batter'], as_index=False)
            .size().rename(columns={'size': '행수'}))


# WBSC narrative 는 타점이 난 타석에 "1 RBI." 처럼 명시해 준다. 구조화 필드(rbi)와
# 출처가 달라서 서로 교차검증이 된다.
_RE_RBI = re.compile(r'(\d+)\s+RBI')


def check_rbi_narrative(pbp):
    """I-1. 영문 원문이 밝힌 타점 vs PBP 의 RBI 열.

    F 의 RBI 는 WBSC 의 플레이별 rbi 를 더해 WBSC 의 boxScore rbi 와 맞추는
    것이라 둘 다 같은 출처다 — 타자 오배정이나 행 중복은 잡아도 값 자체가 옳은지는
    검증하지 못한다. 원문 텍스트는 출처가 달라서 이 대조가 의미를 갖는다.

    타석 행만 본다 — 주자아웃 행은 플레이 하나를 두 행으로 쪼갠 것이라 타석 행과
    같은 원문을 공유해서 타점이 중복으로 읽힌다.
    원문이 타점을 말하지 않은 행은 비교 대상이 아니다(2025 대회에서 1건: 실책이
    끼어 2타점인데 "2 RBI" 문구가 붙지 않았다)."""
    d = pbp[pbp['plate_seq'].notna() & (pbp['result'] != '주자아웃')].copy()
    d['en_RBI'] = (d['result_en'].fillna('').astype(str)
                   .str.extract(_RE_RBI, expand=False).astype(float))
    d['pbp_RBI'] = pd.to_numeric(d['RBI'], errors='coerce').fillna(0)
    cmp_ = d[d['en_RBI'].notna()]
    bad = cmp_[cmp_['en_RBI'] != cmp_['pbp_RBI']]
    return (bad[['file_name', 'game_id', 'team', 'inning', 'topbot', 'batter',
                 'result', 'pbp_RBI', 'en_RBI', 'result_en']], len(cmp_))


def check_runs_narrative(pbp):
    """I-2. 영문 원문의 득점자 수 vs PBP 의 득점 열.

    원문은 득점한 주자를 "이름 scores." 로 한 명씩 적는다. 홈런은 타자 본인이
    거기 안 적히므로 한 명 더한다. away_score/home_score 는 크롤러가 WBSC 의 runs
    필드로 채우는 값이라, 텍스트와 맞춰보면 득점이 엉뚱한 행에 붙는 사고를 잡는다.

    비교는 행이 아니라 **원문 단위**로 한다. 한 투구를 여러 행으로 쪼개는 자리가
    둘 있는데(야수선택 뒤의 주자아웃 행, 도루 뒤 폭투처럼 겹친 주루 사건), 그
    행들은 같은 원문을 공유한다. 행마다 세면 득점을 실제로 받은 한 행만 맞고
    나머지가 전부 불일치로 잡힌다 — 득점 총합은 멀쩡한데도 그렇다."""
    d = pbp.copy()
    en = d['result_en'].fillna('').astype(str)
    d['_en'] = en
    d['en_runs'] = (en.str.count(r'\bscores\b')
                    + d['result'].astype(str).str.contains('홈', na=False).astype(int))
    d['pbp_runs'] = d['away_score'].fillna(0) + d['home_score'].fillna(0)

    key = [k for k in ['file_name', 'game_id', 'inning', 'topbot', '_en'] if k in d.columns]
    grp = d.groupby(key, as_index=False).agg(
        en_runs=('en_runs', 'max'),        # 같은 원문이니 한 값이다
        pbp_runs=('pbp_runs', 'sum'),      # 쪼갠 행에 나뉘어 붙었을 수 있다
        result=('result', lambda x: '/'.join(dict.fromkeys(x.astype(str)))),
        team=('team', 'first'),
        batter=('batter', 'first'),
    )
    bad = grp[grp['en_runs'] != grp['pbp_runs']]
    cols = [c for c in ['file_name', 'game_id', 'team', 'inning', 'topbot', 'batter',
                        'result', 'pbp_runs', 'en_runs', '_en'] if c in bad.columns]
    return bad[cols].rename(columns={'_en': 'result_en'}), len(grp)


In [ ]:
# ════════════════════════════════════════════
# 메인 실행
# ════════════════════════════════════════════
if pbp.empty:
    print("wbsc/{YYYY}_{MM}_global_{letter}.xlsx 가 없습니다 — 먼저 WBSC 크롤링.ipynb를 돌리세요.")
elif actual_inning.empty:
    print(f"wbsc/inning_summary_global_{YEAR}.xlsx 등 실제값 파일이 없습니다 — "
          f"먼저 WBSC 크롤링.ipynb를 돌려서 실제값 파일부터 만드세요.")
else:
    _pbp_games = set(pbp['game_id'].unique())
    _act_games = set(actual_inning['game_id'].unique())
    if _pbp_games ^ _act_games:
        print(f"⚠ PBP에만 있는 경기: {sorted(_pbp_games - _act_games)}")
        print(f"⚠ 실제값에만 있는 경기: {sorted(_act_games - _pbp_games)}")
    # 두 쪽 모두 있는 경기만 대조 (한쪽만 있으면 통째로 불일치로 잡혀 노이즈가 된다)
    _both = _pbp_games & _act_games
    p = pbp[pbp['game_id'].isin(_both)]
    ai = actual_inning[actual_inning['game_id'].isin(_both)]
    ab_ = actual_batter[actual_batter['game_id'].isin(_both)]
    ap = actual_pitcher[actual_pitcher['game_id'].isin(_both)]

    print("\n" + "=" * 60)
    print(f"검수 대상 {len(_both)}경기 / PBP {len(p)}행")
    print("=" * 60)

    # ── A. 결과코드 어휘 ──
    v = check_vocab(p)
    print(f"\n[A] KBSA 어휘에 없는 결과코드: {len(v)}행")
    if len(v):
        print(v.to_string())

    # ── B. 번역 신뢰도 ──
    cf = check_confidence(p)
    print(f"\n[B] 번역 신뢰도 high 아님: {len(cf)}행 "
          f"{p['result_confidence'].value_counts().to_dict()}")
    if len(cf):
        save_diff_xlsx(cf.assign(_d=0), OUT / f"review_confidence_global_{YEAR}.xlsx", [])
        print(f"    -> review_confidence_global_{YEAR}.xlsx")

    # ── C. 타자 이름 정합성 ──
    nmb = check_batter_name(p)
    print(f"\n[C] result_en 선두 이름 != batter: {len(nmb)}행 "
          f"(WBSC batterid stale 버그 탐지용)")
    if len(nmb):
        print(nmb.to_string())

    # ── D. 타순 순환 ──
    rot = check_order_rotation(p)
    print(f"\n[D] 타순 순환 이상: {len(rot)}행")
    if len(rot):
        print(rot.to_string())

    # ── E. 이닝별 아웃/득점 ──
    idf = check_inning(p, ai)
    bad_i = idf[(idf['out_diff'] != 0) | (idf['score_diff'] != 0)]
    save_diff_xlsx(idf, OUT / f"inning_summary_diff_global_{YEAR}.xlsx",
                   ["score_diff", "out_diff"])
    print(f"\n[E] 이닝별 아웃/득점: {len(idf)}행 중 불일치 {len(bad_i)}행 "
          f"-> inning_summary_diff_global_{YEAR}.xlsx")
    if len(bad_i):
        print(bad_i.to_string())

    # ── F. 타자 성적 ──
    bdf = check_batter_box(p, ab_)
    bad_b = bdf[(bdf['AB_diff'] != 0) | (bdf['H_diff'] != 0) | (bdf['RBI_diff'] != 0)]
    save_diff_xlsx(bdf, OUT / f"batter_diff_global_{YEAR}.xlsx",
                   ["AB_diff", "H_diff", "RBI_diff"])
    print(f"\n[F] 타자 AB/H/RBI (한글 코드 재도출): {len(bdf)}행 중 불일치 {len(bad_b)}행 "
          f"-> batter_diff_global_{YEAR}.xlsx")
    if len(bad_b):
        print(bad_b.to_string())

    # ── G. 투수 상대타자수 ──
    pdf = check_pitcher_box(p, ap)
    bad_p = pdf[pdf['BF_diff'] != 0]
    save_diff_xlsx(pdf, OUT / f"pitcher_diff_global_{YEAR}.xlsx", ["BF_diff"])
    print(f"\n[G] 투수 BF: {len(pdf)}행 중 불일치 {len(bad_p)}행 "
          f"-> pitcher_diff_global_{YEAR}.xlsx")
    if len(bad_p):
        print(bad_p.to_string())

    # ── H. 한국 선수 이름 매핑 ──
    kor = check_korea_names(p)
    print(f"\n[H] 한글 치환 안 된 한국 선수: {len(kor)}명")
    if len(kor):
        print(kor.to_string())
        print("    -> 그 로마자 표기를 KOREA_NAME_MAP_2026 의 키로 추가하면 된다.")

    # ── I. 타점·득점 원문 대조 ──
    rbi_bad, rbi_n = check_rbi_narrative(p)
    print(f"\n[I-1] 원문이 밝힌 타점 vs RBI 열: {rbi_n}행 중 불일치 {len(rbi_bad)}행")
    if len(rbi_bad):
        print(rbi_bad.to_string(max_colwidth=90))
    run_bad, run_n = check_runs_narrative(p)
    print(f"[I-2] 원문 득점자 수 vs 득점 열: {run_n}건(원문 단위) 중 불일치 {len(run_bad)}건")
    if len(run_bad):
        print(run_bad.to_string(max_colwidth=90))


    print("\n" + "=" * 60)
    _total = (len(v) + len(nmb) + len(rot) + len(bad_i) + len(bad_b) + len(bad_p)
              + len(kor) + len(rbi_bad) + len(run_bad))
    print(f"실오류 후보 합계 {_total}행 (B는 검수 대상일 뿐 오류가 아님)")
    print("실점(R)/자책(ER)의 승계주자 책임 배분과 주자상황까지 검증하려면 이 PBP 파일을")
    print('"2. 아웃카운트 및 점수 체크" / "3. 주자상황 체크" 노트북에 그대로 넣어 돌리면 된다.')
    print("=" * 60)
